<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/6_rerank_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
project_root_dir = "/Users/karimelbana/wbs/rag_project_DSAI/"
sys.path.append(project_root_dir)

# Part 6: Evaluating a Reranker

This guide provides instructions for the next phase of optimising your RAG pipeline: adding and evaluating a **reranker**. A reranker is a powerful component that can significantly improve the quality of the information your chatbot uses to form its answers.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; What is a Reranker?

So far, our RAG system uses a single-stage process for finding information: it retrieves the `top_k` most similar chunks from the vector store and sends them directly to the LLM. This is fast and effective, but we can improve its precision.

A reranker introduces a second, more sophisticated stage to this process.

1.  **Stage 1: Initial Retrieval (Fast & Broad)**: The vector store quickly retrieves a larger set of potentially relevant documents (e.g., the top 10 or 20). This initial search is optimised for speed.

2.  **Stage 2: Reranking (Slow & Precise)**: The reranker then takes this larger set of documents and uses a more powerful model (typically a cross-encoder) to carefully re-assess and re-order them based on their actual relevance to the question. It then passes only the best few (e.g., the top 2 or 5) to the LLM.

The key benefit is a cleaner, more relevant context for the LLM. By filtering out the noise from the initial retrieval, the reranker helps reduce hallucinations and leads to more accurate, focused answers.

---
## 2.&nbsp; Preparing for the Experiment

To test the effect of a reranker, we'll add a new evaluation stage. This involves adding new configurations and a new evaluation function to our framework.

### 2.1 Add Reranker Configurations

First, we need to define the parameters for our reranker experiment.

1.  Open `evaluation/eval_config.py` in your VSCode editor.
2.  Add the following code to the end of the file.

In [ ]:
import pandas as pd
import numpy as np

results_rechunking = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/chunking_evaluation_summary_2025-09-22_20-50-29.csv')
results_rechunking

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall
0,512,50,0.628205,0.611111,0.638889
1,768,115,0.503968,0.305556,0.416667
2,1024,200,0.539394,0.416667,0.638889


In [ ]:
results_rechunking['mean_score'] = results_rechunking.apply(lambda row: np.mean([
    row['faithfulness'], row['context_precision'], row['context_recall']]
    ),axis = 1)
results_rechunking

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall,mean_score
0,512,50,0.628205,0.611111,0.638889,0.626068
1,768,115,0.503968,0.305556,0.416667,0.408730
2,1024,200,0.539394,0.416667,0.638889,0.531650


In [ ]:
best_chunk_size = int(results_rechunking.sort_values(by='mean_score', ascending=False).head(1)['chunk_size'].values[0])
best_chunk_size

512

In [ ]:
best_chunk_overlap = int(results_rechunking.sort_values(by='mean_score', ascending=False).head(1)['chunk_overlap'].values[0])
best_chunk_overlap

50

In [ ]:
# --- Reranker Evaluation ---
# The 'best' chunking strategy found in the previous evaluation stage.
# IMPORTANT: You must update this with the values you found to be optimal.
BEST_CHUNKING_STRATEGY: dict[str, int] = {'size': 512, 'overlap': 50}
# BEST_CHUNKING_STRATEGY: dict[str, int] = {'size': best_chunk_size, 'overlap': best_chunk_overlap}

# The cross-encoder model to be used for reranking.
RERANKER_MODEL_NAME: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# The different reranker configurations to test.
RERANKER_CONFIGS: list[dict[str, int]] = [
    {'retriever_k': 10, 'reranker_n': 2},
    {'retriever_k': 10, 'reranker_n': 5},
    {'retriever_k': 20, 'reranker_n': 5},
]

* `BEST_CHUNKING_STRATEGY`: This is the baseline for our reranker experiment. It's crucial that you update these values to match the optimal `chunk_size` and `chunk_overlap` you discovered in the previous stage.
* `RERANKER_MODEL_NAME`: This defines the specific cross-encoder model we'll use for the precision reranking step.
* `RERANKER_CONFIGS`: This list defines our experiments. `retriever_k` is the number of documents we initially fetch, and `reranker_n` is the number of documents we pass to the LLM after reranking.

> **Why are we keeping the evaluations separate?**
>
> You might notice that the reranker's performance could be influenced by the chunking strategy. While true, testing every combination would create a huge number of experiments, quickly using up your free API credits. By "siloing" the evaluations, we test one variable at a time. This is a pragmatic approach to find improvements without incurring high costs.

### 2.2 Create the Reranker Evaluation Function

Now, let's update `evaluation/run_evaluation.py` with the logic to test our reranker configurations.

1.  **Add New Imports**: Open `evaluation/run_evaluation.py` and add the new imports. The `RetrieverQueryEngine` and `SentenceTransformerRerank` are new tools for this stage.

In [ ]:
# Add to existing llama-index.core imports
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank

# Add the new configs to the import from evaluation.evaluation_config
from evaluation.eval_config import (
    # ... existing imports
    CHUNKING_STRATEGY_CONFIGS,
    BEST_CHUNKING_STRATEGY,
    RERANKER_MODEL_NAME,
    RERANKER_CONFIGS,
)

2.  **Add the Evaluation Function**: Add the following function to the file, placing it after the `evaluate_chunking_strategies` function.

In [ ]:
def evaluate_reranker_strategies(llm: Groq, embed_model: HuggingFaceEmbedding):
    """ Evaluates different reranker settings on top of the best chunking strategy. """
    print("\n--- 🚀 Stage 3: Evaluating Reranker Strategies ---")
    questions, ground_truths = get_eval_data()
    all_results: list[pd.DataFrame] = []

    best_chunk_size: int = BEST_CHUNKING_STRATEGY['size']
    best_chunk_overlap: int = BEST_CHUNKING_STRATEGY['overlap']

    index: VectorStoreIndex = get_or_build_index(
        chunk_size=best_chunk_size, chunk_overlap=best_chunk_overlap, embed_model=embed_model
    )

    for config in RERANKER_CONFIGS:
        retriever_k, reranker_n = config['retriever_k'], config['reranker_n']
        print(f"\n--- Testing Reranker Config: retrieve_k={retriever_k}, rerank_n={reranker_n} ---")

        retriever: VectorIndexRetrieverr = index.as_retriever(similarity_top_k=retriever_k)
        # You'll notice we've switched from the simple `index.as_query_engine()`
        # to the more explicit `RetrieverQueryEngine.from_args()`.
        # This is necessary because a reranker is a type of `node_postprocessor` -
        # a component that runs after the initial retrieval but before the final answer generation.
        # The `RetrieverQueryEngine` gives us the fine-grained control to construct this multi-stage pipeline
        # by explicitly defining the `retriever` and the list of `node_postprocessors`

        reranker: SentenceTransformerRerank = SentenceTransformerRerank(
            top_n=reranker_n, model=RERANKER_MODEL_NAME
        )

        query_engine: RetrieverQueryEngine = RetrieverQueryEngine.from_args(
            retriever=retriever, node_postprocessors=[reranker], llm=llm
        )

        qa_dataset: Dataset = generate_qa_dataset(query_engine, questions, ground_truths)

        print("--- Running Ragas evaluation for reranker... ---")
        result: Dataset = evaluate(
            dataset=qa_dataset, metrics=EVAL_METRICS, raise_exceptions=True
        )

        results_df: pd.DataFrame = result.to_pandas()
        results_df['chunk_size'] = best_chunk_size
        results_df['chunk_overlap'] = best_chunk_overlap
        results_df['retriever_k'] = retriever_k
        results_df['reranker_n'] = reranker_n
        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)
    save_results(final_df, "reranker_evaluation")
    print("--- ✅ Reranker Strategy Evaluation Complete ---")
    return final_df

> **Why the `RetrieverQueryEngine`?**
>
> You'll notice we've switched from the simple `index.as_query_engine()` to the more explicit `RetrieverQueryEngine.from_args()`. This is necessary because a reranker is a type of `node_postprocessor` - a component that runs after the initial retrieval but before the final answer generation. The `RetrieverQueryEngine` gives us the fine-grained control to construct this multi-stage pipeline by explicitly defining the `retriever` and the list of `node_postprocessors`.

3.  **Update the `save_results` Helper**: To get a correct summary file, we must tell our `save_results` function about the new parameters. Find the function and update the `param_cols` list.

In [ ]:
# In the save_results function:
param_cols: list[str] = [
    col
    for col in ['chunk_size', 'chunk_overlap', 'retriever_k', 'reranker_n']
    if col in results_df.columns
]

### 2.3 Update the Main Execution Block

Finally, let's update the main execution block to run our new stage.

In `evaluation/run_evaluation.py`, update the `if __name__ == "__main__":` block:

In [ ]:
# --- Main execution block ---
if __name__ == "__main__":
    llm_to_test: Groq = initialise_llm()
    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    # Comment out previous stages
    # evaluate_baseline(llm=llm_to_test, embed_model=embed_model_to_test)
    # evaluate_chunking_strategies(llm=llm_to_test, embed_model=embed_model_to_test)

    # Run Stage 3: Reranker Strategy Evaluation
    evaluate_reranker_strategies(llm=llm_to_test, embed_model=embed_model_to_test)

---
## 3.&nbsp; Run the Reranker Evaluation

With the new function and configurations in place, you are ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python -m evaluation.run_evaluation

The first time you run this, it will download the reranker model, which may take a moment. Since you've commented out the previous stages, the script will proceed directly to the reranker evaluation.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find the new `reranker_evaluation_summary_... .csv` file.

Open this summary file. Your goal is to see how the reranker impacts performance. Does increasing the number of documents retrieved (`retriever_k`) and then filtering them down with the reranker (`reranker_n`) improve the final scores? Pay close attention to **`context_precision`**, as this metric should be most directly improved by a good reranker.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You now have a complete, three-stage evaluation framework. The final step is to apply these techniques to your own project and then integrate the best-performing components into your main chatbot.

### 1. Find Your Optimal Reranker Strategy

It's time to run the reranker evaluation on your custom RAG system and analyse the results to find the best settings for your specific documents.

**Your Mission:**

1.  **Confirm Your Baseline**: Double-check that `BEST_CHUNKING_STRATEGY` in `evaluation_config.py` is set to the optimal values you found for your project.
2.  **Run the Reranker Evaluation**: Execute `python -m evaluation.run_evaluation`.
3.  **Set Up Your Analysis Notebook**: In your `notebooks` directory, create a new Notebook file named `03_reranker_analysis.ipynb`.
4.  **Load and Analyse Your Results**:
    * In the new notebook, load the summary results from your reranker experiment.
    * Analyse the results. Which configuration of `retriever_k` and `reranker_n` gives the best scores? Does the reranker provide a significant improvement over your best chunking-only strategy?
    * Document your conclusions in the notebook with text and plots.

### 2. Deeper Experimentation & Final Integration

Now that you have a feel for the process, you can conduct more advanced experiments.

**Challenge:**

* **Experiment with `retriever_k`**: How does retrieving many more documents initially (e.g., `retriever_k: 50`) affect the reranker's performance and the final scores? Is there a point of diminishing returns?
* **Try Different Reranker Models**: The `cross-encoder/ms-marco-MiniLM-L-6-v2` model is small and fast. You can find more powerful (but slower) models on the Hugging Face Hub. Try swapping one in and see if it improves your scores.


In [ ]:
import pandas as pd
import numpy as np

results_rechunking_then_reranker = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/reranker_evaluation_summary_2025-09-22_21-47-10.csv')
results_rechunking_then_reranker

,chunk_size,chunk_overlap,retriever_k,reranker_n,faithfulness,context_precision,context_recall
0,512,50,10,2,0.774510,0.750000,0.750000
1,512,50,10,5,0.819444,0.741667,0.916667
2,512,50,20,5,0.675926,0.741667,0.750000


In [ ]:
results_rechunking_then_reranker['mean_score'] = results_rechunking_then_reranker.apply(lambda row: np.mean([
    row['faithfulness'], row['context_precision'], row['context_recall']]
    ),axis = 1)
results_rechunking_then_reranker

,chunk_size,chunk_overlap,retriever_k,reranker_n,faithfulness,context_precision,context_recall,mean_score
0,512,50,10,2,0.774510,0.750000,0.750000,0.758170
1,512,50,10,5,0.819444,0.741667,0.916667,0.825926
2,512,50,20,5,0.675926,0.741667,0.750000,0.722531


In [ ]:
best_retriever_k = int(results_rechunking_then_reranker.sort_values(by='mean_score', ascending=False).head(1)['retriever_k'].values[0])
best_retriever_k

10

In [ ]:
best_reranker_n = int(results_rechunking_then_reranker.sort_values(by='mean_score', ascending=False).head(1)['reranker_n'].values[0])
best_reranker_n

5

In [ ]:
results_rechunking

,chunk_size,chunk_overlap,faithfulness,context_precision,context_recall,mean_score
0,512,50,0.628205,0.611111,0.638889,0.626068
1,768,115,0.503968,0.305556,0.416667,0.408730
2,1024,200,0.539394,0.416667,0.638889,0.531650


In [ ]:
results_rechunking_then_reranker

,chunk_size,chunk_overlap,retriever_k,reranker_n,faithfulness,context_precision,context_recall,mean_score
0,512,50,10,2,0.774510,0.750000,0.750000,0.758170
1,512,50,10,5,0.819444,0.741667,0.916667,0.825926
2,512,50,20,5,0.675926,0.741667,0.750000,0.722531
